# Phase 4.4 — Publication Visualization

9 figures, each saved as PDF (300 DPI) + PNG (150 DPI), color-blind-safe
palette (matplotlib viridis / colorbrewer Paired).

Figures:
1. Profit curves per anchor scenario
2. Cutoff gap vs PD discrimination quality (Gini)
3. op_cost vs k* per anchor
4. ASB vs Lifetime profit by tenor
5. Bootstrap CI distributions (4-panel density plot)
6. Calibration reliability diagrams (4-panel, base PD models)
7. Feature importance comparison (top 20)
8. Combined stress matrix heatmap
9. Sensitivity hierarchy (parameter spread on k*)

In [1]:
import sys, time, json, joblib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # non-interactive
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

ECO_DIR = REPO_ROOT / "artifacts/economic_framework"
PDV_DIR = REPO_ROOT / "artifacts/pd_quality_stress"
OPC_DIR = REPO_ROOT / "artifacts/op_cost_robustness"
CAL_DIR = REPO_ROOT / "artifacts/calibration_verification"
PD_MODEL = REPO_ROOT / "artifacts/pd_model"
OUT_DIR = REPO_ROOT / "artifacts/figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Color-blind-safe palette (ColorBrewer Paired-like)
PALETTE = {
    "optimistic_base":            "#1f77b4",  # blue
    "realistic_central_boundary": "#2ca02c",  # green
    "moderate_interior":          "#ff7f0e",  # orange
    "adverse_stress":             "#d62728",  # red
}
ANCHOR_LABELS = {
    "optimistic_base":            "Optimistic base",
    "realistic_central_boundary": "Realistic central (boundary)",
    "moderate_interior":          "Moderate interior",
    "adverse_stress":             "Adverse stress",
}

# Common figure setup
plt.rcParams.update({
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "legend.fontsize": 9, "figure.dpi": 100,
    "savefig.dpi": 300, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
})

def save_fig(fig, name):
    pdf_path = OUT_DIR / f"{name}.pdf"
    png_path = OUT_DIR / f"{name}.png"
    fig.savefig(pdf_path)
    fig.savefig(png_path, dpi=150)
    print(f"  Saved: {pdf_path.name} + {png_path.name}")
    plt.close(fig)

T0 = time.time()

## Figure 1 — Profit curves per anchor scenario

In [2]:
# Recreate profit curves: cumulative profit as we approve from lowest PD upward
eco = pd.read_parquet(ECO_DIR / "economics_per_account.parquet")
oot = eco[eco["split_new"] == "oot"].copy().reset_index(drop=True)
y = oot["default_flag_12m"].astype(int).to_numpy()

# Need to recompute per-anchor profit on OOT for this plot
import sys; sys.path.insert(0, str(REPO_ROOT))
from src.economics import (
    batch_lifetime_economics, apr_tier_lookup_vec, apr_tier_lookup_capped_vec,
)

ANCHORS = {
    "optimistic_base": {"pd_multiplier":1.0,"cost_of_funds_annual":0.0,"acquisition_cost":0,"lgd":0.55,"apr_strategy":"tiered_uncapped"},
    "realistic_central_boundary": {"pd_multiplier":2.0,"cost_of_funds_annual":0.03,"acquisition_cost":250,"lgd":0.65,"apr_strategy":"tiered_cap_24"},
    "moderate_interior": {"pd_multiplier":3.0,"cost_of_funds_annual":0.03,"acquisition_cost":250,"lgd":0.65,"apr_strategy":"flat_18"},
    "adverse_stress": {"pd_multiplier":5.0,"cost_of_funds_annual":0.06,"acquisition_cost":500,"lgd":0.85,"apr_strategy":"flat_18"},
}
def apr_array(pd_arr, strategy):
    if strategy == "tiered_uncapped": return apr_tier_lookup_vec(pd_arr)
    if strategy == "tiered_cap_24":   return apr_tier_lookup_capped_vec(pd_arr, cap=0.24)
    if strategy.startswith("flat_"):  return np.full_like(pd_arr, int(strategy.split("_")[1])/100.0)

from sklearn.metrics import roc_curve
pd_raw = oot["pd_lgb_platt"].to_numpy()

fig, ax = plt.subplots(figsize=(9, 6))
youden_per_anchor = {}
for name, a in ANCHORS.items():
    pd_str = np.clip(pd_raw * a["pd_multiplier"], 0.0, 0.999)
    apr_arr = apr_array(pd_str, a["apr_strategy"])
    out = batch_lifetime_economics(
        pd_12m=pd_str, loan_amount=oot["loan_amount"].to_numpy(),
        n_installments=oot["n_installments"].to_numpy(),
        apr=apr_arr, lgd=a["lgd"],
        op_cost_annual=0.0,
        cost_of_funds_annual=a["cost_of_funds_annual"],
        acquisition_cost=float(a["acquisition_cost"]),
    )
    profit = out["Expected_Profit"]
    order = np.argsort(pd_str)
    cum = np.cumsum(profit[order]) / 1e6  # $M
    pct = np.arange(1, len(cum) + 1) / len(cum) * 100
    color = PALETTE[name]
    ax.plot(pct, cum, label=ANCHOR_LABELS[name], color=color, linewidth=1.8)
    # Mark profit-optimal k*
    k_star = int(np.argmax(cum)) + 1 if cum.max() > 0 else 1
    ax.scatter([100*k_star/len(cum)], [cum[k_star-1]], color=color, s=60, zorder=10, marker="o", edgecolor="black", linewidth=0.7)
    # Youden
    fpr, tpr, thr = roc_curve(y, pd_str)
    j_idx = int(np.argmax(tpr - fpr))
    thr_y = float(thr[j_idx])
    youden_per_anchor[name] = thr_y
    accepted_y = pd_str <= thr_y
    if accepted_y.any():
        # Find pct corresponding to youden
        pct_y = float(accepted_y.mean() * 100)
        # Profit at Youden = sum of profit for accepted rows
        prof_y = float(profit[accepted_y].sum()) / 1e6
        ax.scatter([pct_y], [prof_y], color=color, s=80, zorder=10, marker="x", linewidth=2.0)

ax.set_xlabel("Cumulative accepted (% of OOT, sorted by PD ascending)")
ax.set_ylabel("Cumulative Expected Profit ($M)")
ax.set_title("Figure 1 — Profit curves by anchor scenario (OOT, n=64,027)")
ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
ax.legend(loc="upper left", framealpha=0.9)
# Annotation legend for markers
custom = [
    Line2D([0],[0], marker="o", color="black", markerfacecolor="white", markersize=8, linestyle="None", label="profit-optimal k*"),
    Line2D([0],[0], marker="x", color="black", markersize=8, linestyle="None", label="Youden's J"),
]
leg2 = ax.legend(handles=custom, loc="lower right", framealpha=0.9)
ax.add_artist(ax.legend(loc="upper left", framealpha=0.9))
ax.text(0.99, 0.02, "Source: artifacts/economic_framework/economics_per_account.parquet (OOT)",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=7, color="gray")
save_fig(fig, "fig1_profit_curves")

  Saved: fig1_profit_curves.pdf + fig1_profit_curves.png


## Figure 2 — Cutoff gap vs PD discrimination quality

In [3]:
pa = pd.read_csv(PDV_DIR / "cutoffs_by_gini.csv")
fig, ax = plt.subplots(figsize=(9, 6))
for anchor in ANCHORS:
    sub = pa[pa["anchor"] == anchor].sort_values("pd_variant_gini")
    ax.plot(sub["pd_variant_gini"], sub["cutoff_gap"], marker="o",
            linewidth=2, markersize=7, color=PALETTE[anchor],
            label=ANCHOR_LABELS[anchor])
ax.set_xlabel("PD discrimination Gini (achieved)")
ax.set_ylabel("Cutoff gap: profit k* − Youden k (pp)")
ax.set_title("Figure 2 — Profit framework value WIDENS with weaker PD models")
ax.axhline(0, color="gray", linestyle="--", linewidth=0.5)
ax.invert_xaxis()  # weaker → right
ax.legend(loc="upper left", framealpha=0.9)
ax.text(0.99, 0.02, "Source: artifacts/pd_quality_stress/cutoffs_by_gini.csv",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=7, color="gray")
save_fig(fig, "fig2_cutoff_gap_vs_gini")

  Saved: fig2_cutoff_gap_vs_gini.pdf + fig2_cutoff_gap_vs_gini.png


## Figure 3 — op_cost vs k* per anchor

In [4]:
pb = pd.read_csv(OPC_DIR / "cutoffs_by_op_cost.csv")
fig, ax = plt.subplots(figsize=(9, 6))
for anchor in ["realistic_central_boundary", "moderate_interior", "adverse_stress"]:
    sub = pb[pb["anchor"] == anchor].sort_values("op_cost_annual")
    ax.plot(sub["op_cost_annual"] * 100, sub["k_star_approve_pct"],
            marker="s", linewidth=2, markersize=7, color=PALETTE[anchor],
            label=ANCHOR_LABELS[anchor])
ax.axhline(99, color="gray", linestyle=":", linewidth=0.8, label="approve-all threshold (k*=99%)")
ax.axhline(50, color="gray", linestyle="--", linewidth=0.8, label="reject-most threshold (k*=50%)")
ax.set_xlabel("Operating cost (% of loan amount per year)")
ax.set_ylabel("Profit-optimal approval rate k* (%)")
ax.set_title("Figure 3 — op_cost drives profit-optimal cutoff downward, reject-most at adverse + 4%")
ax.set_ylim(-5, 105)
ax.legend(loc="lower left", framealpha=0.9, fontsize=8)
# Annotate the reject-most cell
ax.annotate("REJECT-MOST", xy=(4, 0.31), xytext=(2.7, 18),
            fontsize=9, color="darkred", fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="darkred"))
ax.text(0.99, 0.02, "Source: artifacts/op_cost_robustness/cutoffs_by_op_cost.csv",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=7, color="gray")
save_fig(fig, "fig3_op_cost_vs_kstar")

  Saved: fig3_op_cost_vs_kstar.pdf + fig3_op_cost_vs_kstar.png


## Figure 4 — ASB vs Lifetime profit by tenor

In [5]:
asb = pd.read_csv(ECO_DIR / "asb_comparison.csv")
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(asb))
w = 0.35
ax.bar(x - w/2, asb["LT_total_profit"] / 1e6, width=w, label="Lifetime (locked)",
       color="#2ca02c", edgecolor="black", linewidth=0.4)
ax.bar(x + w/2, asb["ASB_total_profit"] / 1e6, width=w, label="ASB single-period (benchmark)",
       color="#9ecae1", edgecolor="black", linewidth=0.4)
for i, row in asb.iterrows():
    pct = (row["ASB_LT_total_ratio"] - 1) * 100
    ax.text(i + w/2, row["ASB_total_profit"]/1e6 + 5, f"{pct:+.1f}%",
            ha="center", fontsize=9, color="darkblue")
ax.set_xticks(x)
ax.set_xticklabels(asb["tenor"])
ax.set_xlabel("Tenor")
ax.set_ylabel("Total Expected Profit ($M)")
ax.set_title("Figure 4 — ASB single-period UNDERESTIMATES profit, especially for 36m loans")
ax.legend(loc="upper left", framealpha=0.9)
ax.text(0.99, 0.02, "Source: artifacts/economic_framework/asb_comparison.csv",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=7, color="gray")
save_fig(fig, "fig4_asb_vs_lifetime")

  Saved: fig4_asb_vs_lifetime.pdf + fig4_asb_vs_lifetime.png


## Figure 5 — Bootstrap CI distributions (profit_uplift)

In [6]:
bs = pd.read_parquet(ECO_DIR / "bootstrap_anchor_results.parquet")
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=False)
anchor_order = ["optimistic_base","realistic_central_boundary","moderate_interior","adverse_stress"]
for ax, anchor in zip(axes.flat, anchor_order):
    sub = bs[bs["anchor"] == anchor]
    vals = sub["profit_uplift"] / 1e6
    color = PALETTE[anchor]
    ax.hist(vals, bins=30, color=color, alpha=0.7, edgecolor="black", linewidth=0.4)
    p2_5, p50, p97_5 = np.percentile(vals, [2.5, 50, 97.5])
    ax.axvline(p2_5, color="black", linestyle="--", linewidth=0.8, label=f"2.5%: ${p2_5:.2f}M")
    ax.axvline(p50,  color="black", linestyle="-",  linewidth=1.0, label=f"median: ${p50:.2f}M")
    ax.axvline(p97_5,color="black", linestyle="--", linewidth=0.8, label=f"97.5%: ${p97_5:.2f}M")
    ax.axvline(0, color="red", linestyle=":", linewidth=0.8)
    ax.set_title(ANCHOR_LABELS[anchor], color=color, fontweight="bold")
    ax.set_xlabel("Profit uplift over Youden's J ($M)")
    ax.set_ylabel("Bootstrap resample frequency")
    ax.legend(fontsize=7, framealpha=0.9, loc="upper left")
fig.suptitle("Figure 5 — Bootstrap CI distributions of profit uplift (1000 OOT resamples)", fontsize=12, y=1.00)
fig.tight_layout()
fig.text(0.99, 0.005, "Source: artifacts/economic_framework/bootstrap_anchor_results.parquet (OOT, n=64,027 stratified)",
         ha="right", va="bottom", fontsize=7, color="gray")
save_fig(fig, "fig5_bootstrap_ci_density")

  Saved: fig5_bootstrap_ci_density.pdf + fig5_bootstrap_ci_density.png


## Figure 6 — Calibration reliability diagrams (4 panels)

In [7]:
rel = pd.read_csv(CAL_DIR / "reliability_data.csv")
panels = [
    ("LightGBM Platt (PRIMARY)", "LightGBM tuned + Platt"),
    ("LR full-F6E Platt",        "LR full-F6E + Platt"),
    ("LR no-F6E Platt",          "LR no-F6E + Platt"),
    ("Scorecard no-F6E Platt",   "Scorecard no-F6E + Platt"),
]
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, (key, title) in zip(axes.flat, panels):
    sub = rel[rel["pd_source"] == key]
    if len(sub) == 0:
        ax.set_title(f"{title} (DATA MISSING)")
        continue
    # Reliability diagram: x = mean_pred, y = obs_rate
    ax.plot([0, sub["mean_pred"].max()*1.1], [0, sub["mean_pred"].max()*1.1], color="gray", linestyle="--", linewidth=0.7, label="perfect")
    ax.plot(sub["mean_pred"], sub["obs_rate"], marker="o", linewidth=1.6, color="#1f77b4", label="observed")
    ax.fill_between(sub["mean_pred"], sub["obs_rate"], sub["mean_pred"], color="#1f77b4", alpha=0.15)
    ax.set_xlabel("Mean predicted PD (bin)")
    ax.set_ylabel("Observed default rate (bin)")
    ax.set_title(title)
    ax.legend(fontsize=8, loc="upper left")
fig.suptitle("Figure 6 — Calibration reliability diagrams (10 deciles, OOT)", y=1.00)
fig.tight_layout()
fig.text(0.99, 0.005, "Source: artifacts/calibration_verification/reliability_data.csv",
         ha="right", va="bottom", fontsize=7, color="gray")
save_fig(fig, "fig6_reliability_diagrams")

  Saved: fig6_reliability_diagrams.pdf + fig6_reliability_diagrams.png


## Figure 7 — Feature importance comparison (top 20)

In [8]:
# LightGBM gain
lgb_imp = pd.read_csv(PD_MODEL / "lightgbm_feature_importance.csv").sort_values("importance_gain", ascending=False).head(20)
# LR full-F6E coefficients (absolute value, normalized for visual)
lr_coef = pd.read_csv(REPO_ROOT / "artifacts/phase2_rerun_v2/final_coefficients.csv")
lr_coef = lr_coef[lr_coef["feature"] != "const"].copy()
lr_coef["abs_coef"] = lr_coef["coef"].abs()
lr_coef = lr_coef.sort_values("abs_coef", ascending=False).head(20)

fig, axes = plt.subplots(1, 2, figsize=(13, 7))
# LightGBM
ax = axes[0]
ax.barh(lgb_imp["feature"][::-1], lgb_imp["importance_gain"][::-1] / 1e3,
        color="#1f77b4", edgecolor="black", linewidth=0.3)
ax.set_xlabel("Total split gain (×10³)")
ax.set_title("LightGBM tuned — top 20 by gain")
# LR full-F6E
ax = axes[1]
colors = ["#2ca02c" if c > 0 else "#d62728" for c in lr_coef["coef"][::-1]]
ax.barh(lr_coef["feature"][::-1], lr_coef["coef"][::-1], color=colors, edgecolor="black", linewidth=0.3)
ax.axvline(0, color="black", linewidth=0.5)
ax.set_xlabel("Logit coefficient (signed)")
ax.set_title("LR full-F6E — final 22 features (signed coef)")
fig.suptitle("Figure 7 — Feature importance: LightGBM (gain) vs LR (signed coefficient)", y=1.00)
fig.tight_layout()
fig.text(0.99, 0.005, "Source: artifacts/pd_model/lightgbm_feature_importance.csv + artifacts/phase2_rerun_v2/final_coefficients.csv",
         ha="right", va="bottom", fontsize=7, color="gray")
save_fig(fig, "fig7_feature_importance")

  Saved: fig7_feature_importance.pdf + fig7_feature_importance.png


## Figure 8 — Combined stress matrix heatmap

In [9]:
pc = pd.read_csv(ECO_DIR / "phase4_2_combined_grid.csv")
# Build heatmap rows: scenario × Gini, color = profit_uplift in $M
# Use op_cost as facet (3 panels)
op_levels = sorted(pc["op_cost_annual"].unique())
gini_order = ["raw","gini_60","gini_45","gini_30"]
scn_order  = ["realistic_central_boundary","moderate_interior","adverse_stress"]
scn_labels = ["realistic","moderate","adverse"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)
import matplotlib.colors as mcolors
vmin = pc["profit_uplift"].min() / 1e6
vmax = pc["profit_uplift"].max() / 1e6
norm = mcolors.TwoSlopeNorm(vmin=min(vmin, -1), vcenter=0, vmax=max(vmax, 1))

for ax, op in zip(axes, op_levels):
    sub = pc[pc["op_cost_annual"] == op]
    mat = np.zeros((len(scn_order), len(gini_order)))
    for i, sc in enumerate(scn_order):
        for j, g in enumerate(gini_order):
            r = sub[(sub["anchor"] == sc) & (sub["pd_variant"] == g)]
            mat[i, j] = float(r["profit_uplift"].iloc[0]) / 1e6 if len(r) else 0.0
    im = ax.imshow(mat, cmap="RdYlGn", norm=norm, aspect="auto")
    ax.set_xticks(range(len(gini_order)))
    ax.set_xticklabels([f"raw\n(0.80)" if g=="raw" else g.replace("gini_","Gini\n0.").replace("0.6","0.60").replace("0.4","0.45").replace("0.3","0.30") for g in gini_order], fontsize=8)
    ax.set_yticks(range(len(scn_order)))
    ax.set_yticklabels(scn_labels, fontsize=9)
    ax.set_title(f"op_cost = {op*100:.0f}%")
    for i in range(len(scn_order)):
        for j in range(len(gini_order)):
            ax.text(j, i, f"${mat[i,j]:.1f}M", ha="center", va="center",
                    fontsize=8, color="black", fontweight="bold")
fig.colorbar(im, ax=axes, fraction=0.025, label="Profit uplift over Youden ($M)")
fig.suptitle("Figure 8 — Combined stress matrix: profit uplift by scenario × Gini × op_cost", y=1.02)
fig.text(0.99, -0.02, "Source: artifacts/economic_framework/phase4_2_combined_grid.csv",
         ha="right", va="bottom", fontsize=7, color="gray")
save_fig(fig, "fig8_stress_heatmap")

  Saved: fig8_stress_heatmap.pdf + fig8_stress_heatmap.png


## Figure 9 — Sensitivity hierarchy

In [10]:
# From Phase 3.2 stress grid - parameter spread on k*
sens_data = [
    ("PD multiplier",  3.54),
    ("APR strategy",   2.32),
    ("LGD",            1.28),
    ("Acquisition",    0.43),
    ("Cost of funds",  0.43),
]
labels, spreads = zip(*sens_data)
fig, ax = plt.subplots(figsize=(8, 5))
y_pos = np.arange(len(labels))
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(labels)))
ax.barh(y_pos, spreads, color=colors, edgecolor="black", linewidth=0.4)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel("Spread on k* (percentage points across grid range)")
ax.set_title("Figure 9 — Sensitivity hierarchy: PD multiplier and APR strategy dominate cutoff variation")
for i, v in enumerate(spreads):
    ax.text(v + 0.05, i, f"{v:.2f} pp", va="center", fontsize=9)
ax.set_xlim(0, 4.2)
ax.text(0.99, 0.02, "Source: Phase 3.2 stress grid driver analysis",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=7, color="gray")
save_fig(fig, "fig9_sensitivity_hierarchy")
print(f"\nTotal viz wall: {time.time() - T0:.1f}s")

  Saved: fig9_sensitivity_hierarchy.pdf + fig9_sensitivity_hierarchy.png

Total viz wall: 9.9s
